# HireMind AI - Resume Dataset Cleaning Pipeline

This notebook cleans the raw dataset by removing HTML code, stripping punctuation, normalising whitespaces, and preparing text for training.

In [ ]:
import os
import pandas as pd
import re
import string

# Paths
RAW_DATA_PATH = os.path.join("data", "raw", "Resume.csv")
PROCESSED_DATA_PATH = os.path.join("data", "processed", "cleaned_resumes.csv")

print("Verifying raw dataset path:", RAW_DATA_PATH)

## 1. Load Raw Dataset

In [ ]:
df = pd.read_csv(RAW_DATA_PATH)
print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns.")
print("Columns:", df.columns.tolist())
df.head(2)

## 2. Drop Unwanted Columns (Raw HTML)

The column `Resume_html` contains raw HTML markup which is extremely heavy and unnecessary for text classification models.

In [ ]:
if "Resume_html" in df.columns:
    df = df.drop(columns=["Resume_html"])
    print("Successfully dropped Resume_html column.")
print("Remaining columns:", df.columns.tolist())

## 3. Clean Text Content

We clean the raw resume strings by lowercasing, removing urls/emails, stripping punctuation, and clearing double whitespaces.

In [ ]:
def clean_resume_text(text):
    if not isinstance(text, str):
        return ""
    
    # Lowercase text
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    
    # Remove emails
    text = re.sub(r'\b[a-za-z0-9._%+-]+@[a-za-z0-9.-]+\.[a-za-z]{2,}\b', '', text)
    
    # Replace newlines and tabs with spaces
    text = re.sub(r'\s+', ' ', text)
    
    # Strip punctuation and numbers
    text = text.translate(str.maketrans('', '', string.punctuation + string.digits))
    
    return text.strip()

df["Resume_str"] = df["Resume_str"].apply(clean_resume_text)
print("Sample cleaned text preview:")
print(df["Resume_str"].iloc[0][:300])

## 4. Save Cleaned Dataset

Save the lightweight, preprocessed text data to the processed data folder.

In [ ]:
os.makedirs(os.path.dirname(PROCESSED_DATA_PATH), exist_ok=True)
df.to_csv(PROCESSED_DATA_PATH, index=False)
print(f"Successfully saved clean dataset to {PROCESSED_DATA_PATH}")